<div style="width: 100%; text-align: center;">
    <div style="background-color:#007F00; padding: 0.5rem;">
        <h1 style="font-weight: bold; font-size: 2.5em; color: black;"> AGRHYMET CENTRE CLIMATIQUE REGIONAL POUR L'AFRIQUE DE L'OUEST ET LE SAHEL</h1>
   </div>

   <div style="text-align: center;">
  <img src="https://www.sareco.org/wp-content/uploads/2017/07/plrDvYX1.jpg" width="200">
</div>

<a id="1"></a>
### <p style="padding:10px;background-color:#000000 ;margin:0;color:#007F00;font-family:#newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500"> Sélection automatique des sous-bassins par zone d'étude </p>

## Objectif

Éviter de saisir les points d'intérêt à la main : à partir d'une **zone
d'étude** (un pays -- code ISO3 -- ou votre propre shapefile/GeoPackage),
sélectionner automatiquement les **sous-bassins HydroBASINS niveau 5** dont
l'exutoire (ou le polygone, selon la méthode) tombe dans cette zone, puis
enchaîner **tout le reste de la chaîne sans aucune modification** :
téléchargement historique + prévision, seuils de quantile, classification du
risque, cartes journalières -- cadrées sur la zone d'étude.

Ce notebook complète `download_glofas_data.ipynb` et `previsions_risque.ipynb` :
il ne les remplace pas, il ajoute simplement une étape 0 (`glofas_basins.py`)
qui **génère** le fichier de points (`ID,LONG,LAT`) que ces deux notebooks
attendent en entrée. Une fois ce fichier généré, vous pouvez aussi bien
continuer ici que reprendre dans les notebooks existants.

**Couches statiques nécessaires** (dossier `static/`, fournies avec l'atelier) :

- `afrique.gpkg` : limites administratives des pays d'Afrique (colonne
  `GMI_CNTRY` = code ISO3).
- `hybas_af_lev05_with_outlets.gpkg` (ou `hybas_af_lev05_outlet_coordinates.csv`,
  plus léger mais sans les polygones) : sous-bassins HydroBASINS niveau 5,
  avec les coordonnées de leur exutoire (`OUTLET_LONGITUDE`/`OUTLET_LATITUDE`).

**Dépendance supplémentaire** : `geopandas` (+ `pyogrio`, `shapely`) --
inclus dans `environment.yml`/`requirements.txt` mis à jour ; sinon
`pip install geopandas pyogrio shapely`.

In [ ]:
import sys, os
from pathlib import Path
from datetime import date,timedelta

from glofas_basins import resolve_zone, build_zone_points,select_basins_in_zone, basins_to_points, export_points_csv
from glofas_download import download_glofas_discharge
from glofas_extract import extract_glofas_at_points
from glofas_forecast import download_glofas_forecast, extract_glofas_forecast_at_points
from glofas_risk import compute_historical_thresholds, classify_forecast_risk
from glofas_visualize import (
    build_risk_map, generate_daily_risk_maps,
    build_static_risk_map, generate_daily_static_risk_maps,
    generate_static_risk_map_grid, build_max_severity_map,
)

## Configuration EWDS

Comme dans `download_glofas_data.ipynb`/`previsions_risque.ipynb` : le
GloFAS (historique **et** prévision) est servi par l'EWDS, distinct du CDS
« classique » -- d'où un fichier de configuration `cdsapi` séparé
(`~/.cdsapirc-ewds`) pour ne pas entrer en conflit avec une configuration
CDS déjà en place (ex. pour le CMIP6/S2S). **Cette cellule doit être
exécutée avant toute cellule de téléchargement ci-dessous** (§ 3 et § 5) --
sans elle, `cdsapi` retombe sur `~/.cdsapirc` par défaut, qui pointe vers
le CDS et ne connaît pas les jeux de données GloFAS (erreur
`404 ... dataset cems-glofas-historical not found`).

In [ ]:
ewds_config = Path.home() / ".cdsapirc-ewds"

if not ewds_config.exists():
    raise FileNotFoundError(f"Configuration EWDS introuvable : {ewds_config}")

os.environ["CDSAPI_RC"] = str(ewds_config)
print("Configuration sélectionnée :", ewds_config)

## 1. Définir la zone d'étude

Deux façons, au choix (une seule à la fois) :

- **Code ISO3** : `iso3="CMR"` (Cameroun), `"TCD"` (Tchad), `"NGA"`
  (Nigéria), etc. -- recherché dans la colonne `GMI_CNTRY` de `afrique.gpkg`.
- **Zone personnalisée** : `boundary_path="mon_bassin.shp"` (ou `.gpkg`,
  `.geojson`) -- toutes les entités du fichier sont fusionnées en une seule
  géométrie (peu importe qu'il y en ait une ou plusieurs).

`buffer_km` (optionnel) ajoute une marge autour de la zone, utile pour ne
pas exclure un sous-bassin juste à cheval sur la frontière.

In [ ]:
COUNTRY_NAME = "BFA"
zone = resolve_zone(
    iso3=COUNTRY_NAME,                             # ou : boundary_path="ma_zone.shp"
    countries_path="static/afrique.gpkg",
    buffer_km=0,
)
print("Zone :", zone.label, "--", zone.source)

## 2. Sélectionner les sous-bassins et générer le fichier de points

`method` :

- `"outlet"` (défaut) -- l'exutoire du sous-bassin tombe dans la zone.
  Fonctionne avec le GeoPackage complet **ou** le CSV plat des coordonnées
  d'exutoire (pas besoin des polygones) -- c'est aussi la méthode la plus
  cohérente avec le reste de la chaîne (l'extraction GloFAS se fait
  justement au pixel le plus proche de ce point).
- `"intersects"` -- le polygone du sous-bassin touche la zone (au moins
  partiellement) ; plus permissif, capture les bassins transfrontaliers.
  Nécessite le GeoPackage/shapefile complet (polygones).
- `"within"` -- le polygone du sous-bassin est entièrement contenu dans la
  zone ; plus strict. Nécessite également les polygones.

Le résultat (`resultats/points_zone.csv`) a exactement le format attendu par
`glofas_download`/`glofas_extract`/`glofas_forecast` (colonnes `ID`,
`LONG`, `LAT`).

In [ ]:
selection, table = select_basins_in_zone(
    zone,
    basins="static/hybas_af_lev05_with_outlets.gpkg",  # ou le .csv (method='outlet' uniquement)
    method="intersects",
)
points = basins_to_points(selection, id_col=table.id_col, lon_col=table.lon_col, lat_col=table.lat_col)
export_points_csv(points, f"resultats/points_{COUNTRY_NAME}.csv")
points.head()

## 3. Télécharger et extraire l'historique aux points sélectionnés

Identique à `download_glofas_data.ipynb`, mais avec `resultats/points_zone.csv`
comme fichier de points (plus besoin de le saisir à la main). Adaptez
`area` (nord, ouest, sud, est) et la période aux besoins -- voir
`download_glofas_data.ipynb` pour le détail des options.

In [ ]:
download_glofas_discharge(
    year=range(1980, 1990),                 # à adapter -- l'historique complet est long à télécharger
    area=(15.5, -6, 9, 2.5),            # nord, ouest, sud, est -- large, à resserrer sur votre zone
    output_dir=f"glofas_data/{COUNTRY_NAME}",
)

historique = extract_glofas_at_points(
    points="resultats/points_zone.csv",
    input_dir=f"glofas_data/{COUNTRY_NAME}",
    radius_km=10,
    output=f"resultats/extraction_zone/{COUNTRY_NAME}",
)
historique.head()

## 4. Seuils de risque (quantiles historiques)

`start`/`end` (optionnels) restreignent la période utilisée pour le calcul
des quantiles -- par défaut, toute la période historique disponible est
utilisée.

In [ ]:
seuils = compute_historical_thresholds(
    "resultats/extraction_zone_series.csv",
    basis="annual_max",                          # "daily" ou "annual_max"
    start=None,                             # ex. "1985-01-01" -- None = toute la période disponible
    end=None,                               # ex. "2010-12-31"
    output="resultats/seuils_risque_zone.csv",
)
seuils

## 5. Télécharger et extraire la prévision du jour

Réutilise la maille déjà déterminée par l'extraction historique -- pas de
nouveau recalage.

In [ ]:
download_glofas_forecast(
    issue_date=(date.today() - timedelta(days=1)).isoformat(),
    max_days=15,
    area=(15.0, 8.0, -5.0, 20.0),            # même zone que l'historique
    output_dir="glofas_forecast_data",
)

prevision = extract_glofas_forecast_at_points(
    points_meta="resultats/extraction_zone_points.csv",
    forecast_dir="glofas_forecast_data",
    output="resultats/prevision_series_zone.csv",
)
prevision.head()

## 6. Classer la prévision par rapport aux seuils

In [ ]:
risque = classify_forecast_risk(
    "resultats/prevision_series_zone.csv",
    seuils,
    central_stat="median",
    output="resultats/prevision_risque_zone.csv",
)
risque[["id", "issue_date", "leadtime_hours", "membre_central", "minimum", "maximum", "risque"]]

## 7. Cartes de risque journalières, cadrées sur la zone d'étude

`generate_daily_risk_maps` produit automatiquement **une carte HTML par
échéance disponible** (J+1, J+2, ... jusqu'à l'échéance maximale), avec le
contour de la zone d'étude affiché et la carte cadrée dessus (`zone=zone`).
Une carte à la fois reste possible avec `build_risk_map(..., zone=zone)`.

In [ ]:
cartes = generate_daily_risk_maps(
    risque, "resultats/extraction_zone_points.csv", "resultats/cartes_risque_zone",
    zone=zone,
)
cartes

## 8. Cartes statiques (PNG imprimable, style bulletin CILSS/AGRHYMET)

Version statique des cartes ci-dessus (zones colorées, fond de carte,
flèche du nord, barre d'échelle, légende), plus proche des bulletins
imprimés que les cartes interactives : `build_static_risk_map`,
`generate_daily_static_risk_maps` (une PNG par échéance),
`generate_static_risk_map_grid` (toutes les échéances sur une seule
planche) et `build_max_severity_map` (sévérité **maximale** sur toute --
ou une partie de -- la période de prévision, à l'image des cartes
"Maximum forecast flood hazard severity" du CILSS).

Ici, contrairement à `previsions_risque.ipynb`, on dispose des **vrais
polygones de sous-bassins** (`selection`, issue de la sélection § 2) : en
les passant via `basins=selection, basin_id_col=table.id_col`, chaque
sous-bassin est colorié selon son contour réel plutôt qu'un simple disque
autour de son exutoire.

**Dépendance supplémentaire** : `matplotlib` (généralement déjà présente
avec `geopandas`/Jupyter ; sinon `pip install matplotlib`).

In [ ]:
# Une PNG par échéance disponible, coloriée selon le contour réel des sous-bassins
cartes_png = generate_daily_static_risk_maps(
    risque, "resultats/extraction_zone_points.csv", "resultats/cartes_risque_zone_png",
    basins=selection, basin_id_col=table.id_col, zone=zone,
)
cartes_png

In [ ]:
# Toutes les échéances sur une seule planche (grille + légende partagée)
generate_static_risk_map_grid(
    risque, "resultats/extraction_zone_points.csv",
    "resultats/cartes_risque_zone_png/planche_risque_zone.png",
    basins=selection, basin_id_col=table.id_col, zone=zone,
)

In [ ]:
# Sévérité maximale sur toute la période de prévision disponible
build_max_severity_map(
    risque, "resultats/extraction_zone_points.csv",
    "resultats/cartes_risque_zone_png/risque_maximal_zone.png",
    basins=selection, basin_id_col=table.id_col, zone=zone,
)

---

**Résumé** : `resolve_zone` + `select_basins_in_zone` remplacent la saisie
manuelle d'un fichier de points -- tout le reste (`glofas_download`,
`glofas_extract`, `glofas_forecast`, `glofas_risk`, `glofas_visualize`) est
utilisé exactement comme dans `download_glofas_data.ipynb` et
`previsions_risque.ipynb`, en pointant simplement vers
`resultats/points_zone.csv`.

**Limites à garder en tête** : la sélection `method="outlet"` (défaut) peut
manquer un sous-bassin transfrontalier dont l'exutoire tombe juste à
l'extérieur de la zone (ajustez `buffer_km` ou essayez
`method="intersects"`, si vous disposez du GeoPackage complet avec les
polygones). Les seuils restent des quantiles empiriques, pas des périodes de
retour ajustées statistiquement -- voir les limites déjà notées dans
`previsions_risque.ipynb`.